In [2]:
import os
import sys
import csv
import math
import random
import itertools
import collections

import numpy as np
import torch

from sklearn.metrics import recall_score, f1_score
from sklearn.preprocessing import normalize
from sklearn.manifold import TSNE
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.decomposition import PCA

from scipy.optimize import linear_sum_assignment
from scipy.spatial import distance
from scipy.sparse import lil_matrix

import umap  

import plotly.express as px

from tqdm import tqdm

from functools import lru_cache

from datetime import datetime
import time

2025-04-15 15:47:46.278542: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-04-15 15:47:46.293890: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1744746466.312020  106485 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1744746466.317368  106485 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-04-15 15:47:46.337426: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

In [2]:
MAX_SEQ_LEN = 20000
MIN_SEQ_LEN = 2500
MIN_ABUNDANCE_VALUE = 10
csv.field_size_limit(sys.maxsize)

131072

In [3]:
def set_seed(seed=0):
    random.seed(seed)
    torch.manual_seed(seed)

In [4]:
def create_kmer_mapping(k):
    letters = ['A', 'C', 'G', 'T']
    return {''.join(kmer): idx for idx, kmer in enumerate(itertools.product(letters, repeat=k))}

In [5]:
def get_cooccurrence_counts(file_path, k, window_size, read_sample_size, verbose=False):
    kmer2id = create_kmer_mapping(k)
    vocab_size = 4**k
    counts = np.zeros((vocab_size, vocab_size), dtype=float)

    def seq_to_kmer_ids(seq):
        return [kmer2id[seq[i:i+k]] for i in range(len(seq) - k + 1) if seq[i:i+k] in kmer2id]

    with open(file_path, 'r') as f:
        selected_reads = []
        for i, line in enumerate(f):
            if read_sample_size > 0 and len(selected_reads) >= read_sample_size:
                break
            left, right = line.strip().split(',')
            selected_reads.append(left)
            selected_reads.append(right)

    if verbose:
        print(f"Loaded {len(selected_reads)} reads (from {read_sample_size} pairs)")

    for read in selected_reads:
        kmer_ids = seq_to_kmer_ids(read)
        for i, center_id in enumerate(kmer_ids):
            for j in range(max(0, i - window_size), min(len(kmer_ids), i + window_size + 1)):
                if j != i:
                    context_id = kmer_ids[j]
                    i1, i2 = sorted((center_id, context_id))
                    counts[i1, i2] += 1

    return counts[np.triu_indices(vocab_size, k=1)] / (2 * read_sample_size)

In [5]:
def compute_poisson_loss(embeddings, pairs, counts):
    dist_sq = torch.norm(
        torch.index_select(embeddings, 0, pairs[0]) - 
        torch.index_select(embeddings, 0, pairs[1]), 
        p=2, dim=1
    )**2 
    return 0.5 * (counts * dist_sq + torch.exp(-dist_sq)).sum()

In [6]:
def train_poisson_embeddings(file_path, k=4, dim=256, lr=0.001, epoch_num=1000, 
                             batch_size=0, window_size=4, read_sample_size=10000,
                             device="cpu", verbose=False, seed=0):
    
    print("=================Training Poisson embeddings=================")

    device = torch.device(device)
    set_seed(seed)

    vocab_size = 4**k
    embeddings = torch.nn.Parameter(
        2 * torch.rand((vocab_size, dim), device=device) - 1, requires_grad=True
    )
    optimizer = torch.optim.Adam([embeddings], lr=lr)

    with torch.no_grad():
        pairs = torch.triu_indices(vocab_size, vocab_size, offset=1, device=device)
        counts_np = get_cooccurrence_counts(file_path, k, window_size, read_sample_size, verbose)
        counts = torch.from_numpy(counts_np).to(device)

    if batch_size <= 0 or batch_size > pairs.shape[1]:
        batch_size = pairs.shape[1]  # full batch

    losses = []

    for epoch in range(epoch_num):
        perm = torch.randperm(pairs.shape[1], device=device)
        epoch_loss = 0.0

        if verbose:
            pbar = tqdm(total=math.ceil(pairs.shape[1] / batch_size), desc=f"Epoch {epoch+1}/{epoch_num}", unit="batch")

        for i in range(0, pairs.shape[1], batch_size):
            batch_idx = perm[i:i+batch_size]
            batch_pairs = pairs[:, batch_idx]
            batch_counts = counts[batch_idx]

            optimizer.zero_grad()
            loss = compute_poisson_loss(embeddings, batch_pairs, batch_counts)
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()

            if verbose:
                pbar.update(1)

        epoch_loss /= math.ceil(pairs.shape[1] / batch_size)
        losses.append(epoch_loss)

        if verbose:
            pbar.close()
            print(f"Epoch {epoch+1}, Loss: {epoch_loss:.4f}")
    
    return embeddings, losses

In [7]:
def poisson_sequences_to_embeddings(sequences, embeddings, k):
    print("=================Converting sequences to embeddings=================")
    
    kmer2id = create_kmer_mapping(k)
    vocab_size = 4**k
    device = embeddings.device

    all_profiles = torch.zeros((len(sequences), vocab_size), dtype=torch.float32, device=device)

    for seq_idx, seq in enumerate(sequences):
        for i in range(len(seq) - k + 1):
            kmer = seq[i:i+k]
            if kmer in kmer2id:  # just in case
                all_profiles[seq_idx, kmer2id[kmer]] += 1

    profile_sums = all_profiles.sum(dim=1, keepdim=True).clamp(min=1.0)

    result = (all_profiles @ embeddings) / profile_sums

    return result.detach().cpu().numpy()


In [8]:
@lru_cache(maxsize=16)
def get_kmer2id(k):
    return create_kmer_mapping(k)

def read2kmer_profile(read, k, kmers_num, normalized=False):
    kmer2id = get_kmer2id(k)
    profile = np.zeros(kmers_num, dtype=np.float32)
    
    for i in range(len(read) - k + 1):
        kmer = read[i:i + k]
        if kmer in kmer2id:
            profile[kmer2id[kmer]] += 1
    
    if normalized:
        total = profile.sum()
        if total > 0:
            profile /= total
    return profile

In [9]:
def nonlinear_encoder(kmer_profile, linear1, batch1, activation1, dropout1, linear2):
    output = linear1(kmer_profile)
    output = batch1(output)
    output = activation1(output)
    output = dropout1(output)
    output = linear2(output)
    return output

In [10]:
def nonlinear_loss(left_embeddings, right_embeddings, labels, name="bern"):
    if name == "bern":
        p = torch.exp(-torch.norm(left_embeddings - right_embeddings, p=2, dim=1)**2)
        return torch.nn.functional.binary_cross_entropy(p, labels, reduction='mean')
    elif name == "poisson":
        log_lambda = -torch.norm(left_embeddings - right_embeddings, p=2, dim=1)**2
        return torch.mean(-(labels * log_lambda) + torch.exp(log_lambda))
    elif name == "hinge":
        d = torch.norm(left_embeddings - right_embeddings, p=2, dim=1)
        return torch.mean(labels * (d**2) + (1 - labels) * torch.nn.functional.relu(1 - d)**2)
    else:
        raise ValueError(f"Unknown loss function: {name}")

In [11]:
def train_nonlinear_embeddings(file_path, k=2, dim=256, lr=0.001, epoch_num=100, 
                              batch_size=32, neg_sample_per_pos=1000, max_read_num=10000,
                              loss_name="bern", device="cpu", verbose=False, seed=0):
    
    print("=================Training non-linear embeddings=================")
    
    device = torch.device(device)
    set_seed(seed)
    kmers_num = 4**k

    linear1 = torch.nn.Linear(kmers_num, 512, device=device)
    batch1 = torch.nn.BatchNorm1d(512, device=device)
    activation1 = torch.nn.Sigmoid()
    dropout1 = torch.nn.Dropout(0.2)
    linear2 = torch.nn.Linear(512, dim, device=device)

    optimizer = torch.optim.Adam(
        [*linear1.parameters(), *batch1.parameters(), *linear2.parameters()], lr=lr
    )

    with open(file_path, 'r') as f:
        lines = f.readlines()
    
    if max_read_num > 0:
        lines = random.sample(lines, min(max_read_num, len(lines)))

    transform_func = lambda x: read2kmer_profile(x, k, kmers_num)
    
    left_kmer_profiles = []
    right_kmer_profiles = []
    for line in lines:
        left_read, right_read = line.strip().split(',')
        left_kmer_profiles.append(transform_func(left_read))
        right_kmer_profiles.append(transform_func(right_read))
    
    both_kmer_profiles_np = np.asarray(left_kmer_profiles + right_kmer_profiles, dtype=np.float32)
    both_kmer_profiles = torch.from_numpy(both_kmer_profiles_np).to(device)

    data_size = len(both_kmer_profiles) // 2
    losses = []

    for epoch in tqdm(range(epoch_num), desc="Training Progress", file=sys.stderr):
        indices = torch.randperm(data_size, device=device)
        epoch_loss = 0

        total_neg = neg_sample_per_pos * data_size
        neg_indices = torch.randint(0, len(both_kmer_profiles), (2 * total_neg,), device=device)

        neg_left_all = both_kmer_profiles[neg_indices[:total_neg]]
        neg_right_all = both_kmer_profiles[neg_indices[total_neg:]]

        for i in tqdm(range(0, data_size, batch_size), desc=f"Epoch {epoch+1}", leave=False, file=sys.stderr):
            batch_idx = indices[i:i+batch_size]
            if len(batch_idx) < batch_size:
                continue

            left_batch = both_kmer_profiles[batch_idx]
            right_batch = both_kmer_profiles[batch_idx + data_size]

            start = i * neg_sample_per_pos
            end = start + batch_size * neg_sample_per_pos
            neg_left = neg_left_all[start:end]
            neg_right = neg_right_all[start:end]

            left_batch = torch.cat((left_batch, neg_left), dim=0)
            right_batch = torch.cat((right_batch, neg_right), dim=0)

            labels = torch.cat([
                torch.ones(len(batch_idx), device=device),
                torch.zeros(len(neg_left), device=device)
            ])

            optimizer.zero_grad()
            left_output = nonlinear_encoder(left_batch, linear1, batch1, activation1, dropout1, linear2)
            right_output = nonlinear_encoder(right_batch, linear1, batch1, activation1, dropout1, linear2)
            batch_loss = nonlinear_loss(left_output, right_output, labels, loss_name)
            batch_loss.backward()
            optimizer.step()
            epoch_loss += batch_loss.item()
        
        epoch_loss /= math.ceil(data_size / batch_size)
        losses.append(epoch_loss)

        if verbose:
            print(f"epoch: {epoch}, loss: {epoch_loss}")
            sys.stderr.flush()
    
    return linear1, batch1, activation1, dropout1, linear2, losses


In [ ]:
def nonlinear_sequences_to_embeddings(sequences, k, linear1, batch1, activation1, dropout1, linear2):
    print("=================Converting sequences to embeddings=================")
    
    device = linear1.weight.device
    kmers_num = 4**k
    kmer2id = get_kmer2id(k)  

    profiles = np.zeros((len(sequences), kmers_num), dtype=np.float32)
    for i, seq in enumerate(sequences):
        for j in range(len(seq) - k + 1):
            kmer = seq[j:j+k]
            if kmer in kmer2id:
                profiles[i, kmer2id[kmer]] += 1

    kmer_profiles = torch.from_numpy(profiles).to(torch.float32).to(device)

    with torch.no_grad():
        embeddings = nonlinear_encoder(kmer_profiles, linear1, batch1, activation1, dropout1, linear2)

    return embeddings.cpu().numpy()

In [12]:
@lru_cache(maxsize=16)
def get_tnf_index(k):
    return {''.join(kmer): idx for idx, kmer in enumerate(itertools.product("ATCG", repeat=k))}

def tfidf_umap_embeddings(dna_sequences, k=4, n_components=2):

    print("=================Calculating TNF TF-IDF=================")
    
    tnf_index = get_tnf_index(k)  
    vocab_size = len(tnf_index)
    num_seqs = len(dna_sequences)
    
    counts = np.zeros((num_seqs, vocab_size), dtype=np.float32)
    
    for j, seq in enumerate(dna_sequences):
        for i in range(len(seq) - k + 1):
            kmer = seq[i:i+k]
            if kmer in tnf_index:
                counts[j, tnf_index[kmer]] += 1

    transformer = TfidfTransformer(norm='l2', use_idf=True, smooth_idf=True)
    tfidf_embedding = transformer.fit_transform(counts).toarray()

    print("=================Reducing Dimensionality with UMAP=================")
    
    reducer = umap.UMAP(n_components=n_components, random_state=0)
    umap_embedding = reducer.fit_transform(tfidf_embedding)
    
    return umap_embedding


In [13]:
@lru_cache(maxsize=16)
def get_tnf_index(k):
    return {''.join(kmer): idx for idx, kmer in enumerate(itertools.product("ATCG", repeat=k))}

def calculate_tnf(dna_sequences, k=4):
    print("=================Calculating TNF k-mer profile=================")

    tnf_index = get_tnf_index(k)
    vocab_size = 4**k
    embedding = np.zeros((len(dna_sequences), vocab_size), dtype=np.float32)

    for j, seq in enumerate(dna_sequences):
        for i in range(len(seq) - k + 1):
            kmer = seq[i:i+k]
            if kmer in tnf_index: 
                embedding[j, tnf_index[kmer]] += 1

    return embedding

In [14]:
def kmedoid(features, min_similarity=0.8, min_bin_size=10, max_iter=1000, metric="dot"):
    print(" ============================ Running k-medoids clustering ============================")

    features = features.astype(np.float32)
    n = features.shape[0]
    p = -np.ones(n, dtype=int)

    if metric == "dot":
        similarities = np.dot(features, features.T)
        similarities[similarities < min_similarity] = 0
        row_sum = np.sum(similarities, axis=1)
    else:
        similarities = lil_matrix((n, n), dtype=np.float32)
        for i in range(n):
            if metric == "l2":
                dist = distance.cdist(features[[i]], features, 'euclidean')[0]
            elif metric == "l1":
                dist = distance.cdist(features[[i]], features, 'minkowski', p=1.)[0]
            else:
                raise ValueError("Invalid metric for sparse computation")
            sim = np.exp(-dist)
            similarities.rows[i] = np.where(sim >= min_similarity)[0].tolist()
            similarities.data[i] = sim[sim >= min_similarity].tolist()
        similarities = similarities.tocsr()
        similarities.eliminate_zeros()
        row_sum = np.array(similarities.sum(axis=1)).flatten()

    iter_count = 1
    while np.any(p == -1) and iter_count <= max_iter:
        s = np.argmax(row_sum)
        current_medoid = features[s]
        selected_idx = None

        for _ in range(3):
            if metric == "dot":
                similarity = np.dot(features, current_medoid)
            else:
                if metric == "l2":
                    dist = distance.cdist(features, current_medoid[np.newaxis], 'euclidean').squeeze()
                elif metric == "l1":
                    dist = distance.cdist(features, current_medoid[np.newaxis], 'minkowski', p=1.).squeeze()
                else:
                    raise ValueError("Invalid metric for medoid update")
                similarity = np.exp(-dist)

            idx_within = similarity >= min_similarity
            idx_available = (p == -1)
            selected_idx = np.where(np.logical_and(idx_within, idx_available))[0]

            if len(selected_idx) > 0:
                current_medoid = np.mean(features[selected_idx], axis=0)

        if selected_idx is not None and len(selected_idx) > 0:
            p[selected_idx] = iter_count
            if metric == "dot":
                row_sum -= np.sum(similarities[:, selected_idx], axis=1)
            else:
                selected_mask = np.zeros(n, dtype=bool)
                selected_mask[selected_idx] = True
                row_sum -= similarities[:, selected_mask].sum(axis=1).A1
            row_sum[selected_idx] = 0
        else:
            break

        iter_count += 1

    unique, counts = np.unique(p[p != -1], return_counts=True)
    for label, count in zip(unique, counts):
        if count < min_bin_size:
            p[p == label] = -1

    return p

In [15]:
def align_labels_via_hungarian_algorithm(true_labels, predicted_labels):
    true_labels = np.asarray(true_labels)
    predicted_labels = np.asarray(predicted_labels)

    max_label = max(true_labels.max(), predicted_labels.max()) + 1
    confusion_matrix = np.zeros((max_label, max_label), dtype=np.int32)

    np.add.at(confusion_matrix, (true_labels, predicted_labels), 1)

    row_ind, col_ind = linear_sum_assignment(confusion_matrix, maximize=True)

    return {pred: true for true, pred in zip(row_ind, col_ind)}

In [16]:
def compute_class_center_medium_similarity(embeddings, labels, metric="dot"):
    embeddings = embeddings.astype(np.float32)
    labels = np.asarray(labels)
    
    idx = np.argsort(labels)
    embeddings = embeddings[idx]
    labels = labels[idx]
    n_sample_per_class = np.bincount(labels)

    all_similarities = np.zeros(len(embeddings), dtype=np.float32)
    
    start = 0
    for count in n_sample_per_class:
        end = start + count
        class_emb = embeddings[start:end]
        center = np.mean(class_emb, axis=0)

        if metric == "dot":
            sims = np.dot(class_emb, center)
        elif metric == "l2":
            sims = np.exp(-distance.cdist(class_emb, center[None], 'euclidean')).flatten()
        elif metric == "l1":
            sims = np.exp(-distance.cdist(class_emb, center[None], 'minkowski', p=1)).flatten()
        else:
            raise ValueError("Invalid metric!")

        all_similarities[start:end] = sims
        start = end

    percentiles = np.percentile(all_similarities, [10, 20, 30, 40, 50, 60, 70, 80, 90])
    print("Percentile values:", percentiles.tolist())

    return percentiles.tolist()

In [17]:
def plot_clustering(embedding, true_labels, predicted_labels, model_name, sample, 
                    f1_results, recall_results, thresholds, output_dir="plots", vis_method="tsne"):

    if vis_method == "pca":
        reducer = PCA(n_components=2, random_state=0)
    elif vis_method == "tsne":
        reducer = TSNE(n_components=2, random_state=0, perplexity=30, init='pca')
    elif vis_method == "umap":
        reducer = umap.UMAP(n_components=2, random_state=0)
    else:
        raise ValueError(f"Invalid visualization method: {vis_method}")

    embedding_2d = reducer.fit_transform(embedding)

    os.makedirs(output_dir, exist_ok=True)

    predicted_labels = np.array(predicted_labels)

    fig = px.scatter(
        x=embedding_2d[:, 0],
        y=embedding_2d[:, 1],
        color=predicted_labels.astype(str),
        labels={'x': 'Component 1', 'y': 'Component 2'},
        title=f"{model_name} - Sample {sample} (Predicted Labels)" if f1_results is not None else f"{model_name} - Sample {sample} (True Labels)",
    )

    if f1_results is not None and recall_results is not None:
        f1_str = ", ".join([f"{t}:{f:.2f}" for t, f in zip(thresholds, f1_results)])
        recall_str = ", ".join([f"{t}:{r:.2f}" for t, r in zip(thresholds, recall_results)])
        num_clusters = len(set(predicted_labels) - {-1})
        metrics_text = f"F1: {f1_str}<br>Recall: {recall_str}<br>Clusters: {num_clusters}"
    else:
        num_clusters = len(set(true_labels))  
        metrics_text = f"Clusters (True): {num_clusters}"

    fig.add_annotation(
        text=metrics_text,
        xref="paper", yref="paper",
        x=1.05, y=1,
        showarrow=False,
        align="left",
        bordercolor="black",
        borderwidth=1,
        bgcolor="white",
        opacity=0.8
    )

    output_path = os.path.join(output_dir, f"{model_name}_{vis_method}_sample_{sample}.html")
    fig.write_html(output_path)

In [18]:
def evaluate_binning(data_dir, species="reference", samples=[5], model_name="poisson", 
                     k=2, dim=2, output_file="results.txt", test_model_path=None, 
                     lr=0.001, epoch_num=100, batch_size=32, neg_sample_per_pos=1000, 
                     max_read_num=10000, loss_name="bern", vis_methods = ["pca", "tsne", "umap"], device="cpu", verbose=False, seed=0):

    MAX_SEQ_LEN = 20000
    MIN_SEQ_LEN = 2500
    MIN_ABUNDANCE_VALUE = 10

    all_losses = []
    all_recall_results = {}
    all_f1_results = {}
    thresholds = [0.5, 0.6, 0.7, 0.8, 0.9]

    with open(output_file, 'a') as out_f:
        out_f.write("\n" + "="*50 + "\n")
        out_f.write(f"Evaluation started at: {datetime.now():%d/%m/%Y %H:%M:%S}\n")
    
    for sample in samples:
        print(f"\n--- Evaluating {model_name.upper()} on {species}, Sample {sample} ---")
        metric = "l2" if model_name in ["poisson", "nonlinear", "tfidf_umap"] else "l1" if model_name == "kmerprofile" else None

        clustering_file = os.path.join(data_dir, species, "clustering_0.tsv")
        with open(clustering_file, "r") as f:
            reader = csv.reader(f, delimiter="\t")
            data = list(reader)[1:]
        dna_sequences = [d[0][:MAX_SEQ_LEN] for d in data]
        labels = np.array([d[1] for d in data])
        label2id = {l: i for i, l in enumerate(sorted(set(labels)))}
        labels = np.array([label2id[l] for l in labels])
        print(f"Loaded {len(dna_sequences)} sequences from clustering, {len(label2id)} clusters.")

        emb_start_time = time.time()
        if model_name == "poisson" and test_model_path:
            embeddings, losses = train_poisson_embeddings(test_model_path, k=k, dim=dim, lr=lr, window_size=2, 
                                                          epoch_num=epoch_num, batch_size=batch_size, verbose=verbose, seed=seed)
            all_losses.append(losses)
            embedding_clustering = poisson_sequences_to_embeddings(dna_sequences, embeddings, k)
        elif model_name == "nonlinear" and test_model_path:
            linear1, batch1, activation1, dropout1, linear2, losses = train_nonlinear_embeddings(
                test_model_path, k=k, dim=dim, lr=lr, epoch_num=epoch_num, batch_size=batch_size,
                neg_sample_per_pos=neg_sample_per_pos, max_read_num=max_read_num, loss_name=loss_name,
                device=device, verbose=verbose, seed=seed)
            all_losses.append(losses)
            embedding_clustering = nonlinear_sequences_to_embeddings(dna_sequences, k, linear1, batch1, activation1, dropout1, linear2)
        elif model_name == "kmerprofile":
            embedding_clustering = calculate_tnf(dna_sequences, k=k)
        elif model_name == "tfidf_umap":
            embedding_clustering = tfidf_umap_embeddings(dna_sequences, k=k, n_components=dim)
        else:
            raise ValueError(f"Unsupported model {model_name} without test model path")
        emb_end_time = time.time()
        emb_time = emb_end_time - emb_start_time
        print(f"Embedding creation time (clustering data): {emb_time:.2f} seconds")
        
        print(f"Clustering embedding shape: {embedding_clustering.shape}")
        percentile_values = compute_class_center_medium_similarity(embedding_clustering, labels, metric=metric)
        threshold = percentile_values[-3]  # 70th percentile
        print(f"Similarity threshold: {threshold:.4f}")

        for vis_method in vis_methods:
            plot_clustering(embedding_clustering, labels, labels,
                            f"{model_name}_clustering", sample, None, None, thresholds,
                            output_dir=f"plots/clustring/{species}/{vis_method}/", vis_method=vis_method)

        binning_file = os.path.join(data_dir, species, f"binning_{sample}.tsv")
        with open(binning_file, "r") as f:
            reader = csv.reader(f, delimiter="\t")
            data = list(reader)[1:]
        dna_sequences = [d[0][:MAX_SEQ_LEN] for d in data]
        labels_bin = [d[1] for d in data]

        valid_idx = [i for i, seq in enumerate(dna_sequences) if len(seq) >= MIN_SEQ_LEN]
        dna_sequences = [dna_sequences[i] for i in valid_idx]
        labels_bin = [labels_bin[i] for i in valid_idx]
        label_counts = collections.Counter(labels_bin)
        abundant_idx = [i for i, l in enumerate(labels_bin) if label_counts[l] >= MIN_ABUNDANCE_VALUE]
        dna_sequences = [dna_sequences[i] for i in abundant_idx]
        labels_bin = [labels_bin[i] for i in abundant_idx]
        label2id = {l: i for i, l in enumerate(sorted(set(labels_bin)))}
        labels_bin = np.array([label2id[l] for l in labels_bin])
        print(f"{len(dna_sequences)} sequences remaining, {len(label2id)} clusters.")

        bin_emb_start_time = time.time()
        if model_name == "poisson":
            embedding = poisson_sequences_to_embeddings(dna_sequences, embeddings, k)
        elif model_name == "nonlinear":
            embedding = nonlinear_sequences_to_embeddings(dna_sequences, k, linear1, batch1, activation1, dropout1, linear2)
        elif model_name == "tfidf_umap":
            embedding = tfidf_umap_embeddings(dna_sequences, k=k, n_components=dim)
        elif model_name == "kmerprofile":
            embedding = calculate_tnf(dna_sequences, k=k)
        else:
            raise ValueError(f"Unsupported model {model_name} without test model path")
        bin_emb_end_time = time.time()
        bin_emb_time = bin_emb_end_time - bin_emb_start_time
        print(f"Embedding creation time (binning data): {bin_emb_time:.2f} seconds")

        binning_start_time = time.time()
        binning_results = kmedoid(embedding, min_similarity=threshold, metric=metric)
        valid_mask = binning_results != -1
        predicted_labels = binning_results[valid_mask]
        true_labels_bin = labels_bin[valid_mask]
        binning_end_time = time.time()
        binning_time = binning_end_time - binning_start_time
        print(f"Binning (clustering) execution time: {binning_time:.2f} seconds")

        alignment = align_labels_via_hungarian_algorithm(true_labels_bin, predicted_labels)
        predicted_labels_bin = [alignment[pred] for pred in predicted_labels]
        recall_bin = np.sort(recall_score(true_labels_bin, predicted_labels_bin, average=None, zero_division=0))
        f1_bin = np.sort(f1_score(true_labels_bin, predicted_labels_bin, average=None, zero_division=0))
        recall_results = [np.sum(recall_bin > t) for t in thresholds]
        f1_results = [np.sum(f1_bin > t) for t in thresholds]
        all_recall_results[sample] = recall_results
        all_f1_results[sample] = f1_results

        for vis_method in vis_methods:
            plot_clustering(embedding[valid_mask], true_labels_bin, predicted_labels_bin, 
                            f"{model_name}_binning", sample, f1_results, recall_results, thresholds,
                            output_dir=f"plots/binning/{species}/{vis_method}/", vis_method=vis_method)

        with open(output_file, 'a') as f:
            f.write(f"\n{datetime.now():%d/%m/%Y %H:%M:%S}\n")
            f.write(f"Model: {model_name}, Species: {species}, Sample: {sample}\n")
            f.write(f"Recall: {recall_results}\n")
            f.write(f"F1: {f1_results}\n")
            f.write(f"Threshold: {threshold:.4f}\n")
            f.write(f"Embedding time (clustering data): {emb_time:.2f} sec\n")
            f.write(f"Embedding time (binning data): {bin_emb_time:.2f} sec\n")
            f.write(f"Binning execution time: {binning_time:.2f} sec\n")

    return all_losses, all_recall_results, all_f1_results, thresholds

In [ ]:
def select_loss(model):
    if model == "poisson":
        return "poisson"
    elif model == "nonlinear":
        return "bern" 
    else:
        return None

In [19]:
species_list = ["plant", "marine", "reference"]
model_list = ["poisson", "kmerprofile", "nonlinear", "tfidf_umap"]
vis_methods = ["pca", "tsne", "umap"]

data_dir = "revisitingkmers/dnabert-s_eval/"
samples = [5, 6]  
k = 4

default_dim = 128  

device = "cpu"
test_model_path = "revisitingkmers/dataset_train/datal_48k.csv"


lr = 0.001
epoch_num = 100
batch_size = 64
neg_sample_per_pos = 300
max_read_num = 10000

In [ ]:
results = {}

for species in species_list:
    for model in model_list:
        loss_param = select_loss(model)
        if model == "nonlinear":
            dim_val = 256
        else:
            dim_val = default_dim

        output_file = f"results_{species}_{model}.txt"
        
        print(f"Running evaluation for Species='{species}', Model='{model}'")

        losses, recall_results, f1_results, thresholds = evaluate_binning(
            data_dir, species=species, samples=samples, model_name=model,
            k=k, dim=dim_val, output_file=output_file, test_model_path=test_model_path,
            lr=lr, epoch_num=epoch_num, batch_size=batch_size, neg_sample_per_pos=neg_sample_per_pos,
            max_read_num=max_read_num, loss_name=loss_param,
            device=device, verbose=True, seed=0
        )
        results[(species, model)] = {
            "losses": losses,
            "recall": recall_results,
            "f1": f1_results,
            "thresholds": thresholds
        }
        print(f"Finished evaluation for Species='{species}', Model='{model}'.\n")

print("\n=== Evaluation Summary ===")
for combo, res in results.items():
    species, model = combo
    print(f"Species: {species}, Model: {model}")
    print(f"Thresholds: {res['thresholds']}")
    print(f"Recall: {res['recall']}")
    print(f"F1: {res['f1']}\n")

Running evaluation for Species='plant', Model='tfidf_umap'

--- Evaluating TFIDF_UMAP on plant, Sample 5 ---
Loaded 10800 sequences from clustering, 108 clusters.
=================Calculating TNF TF-IDF=================
=================Reducing Dimensionality with UMAP=================


/home/gandalf/anaconda3/envs/tf/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

/home/gandalf/anaconda3/envs/tf/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.



Embedding creation time (clustering data): 59.13 seconds
Clustering embedding shape: (10800, 128)
Percentile values: [0.17072792649269106, 0.33897984623909, 0.44810711145400994, 0.5184095501899719, 0.5702208876609802, 0.618463921546936, 0.6653921484947205, 0.717543113231659, 0.7819611430168152]
Similarity threshold: 0.6654


/home/gandalf/anaconda3/envs/tf/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

/home/gandalf/anaconda3/envs/tf/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.



71642 sequences remaining, 181 clusters.
=================Calculating TNF TF-IDF=================
=================Reducing Dimensionality with UMAP=================


...

Embedding creation time (clustering data): 57.69 seconds
Clustering embedding shape: (10800, 128)
Percentile values: [0.17072792649269106, 0.33897984623909, 0.44810711145400994, 0.5184095501899719, 0.5702208876609802, 0.618463921546936, 0.6653921484947205, 0.717543113231659, 0.7819611430168152]
Similarity threshold: 0.6654


/home/gandalf/anaconda3/envs/tf/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

/home/gandalf/anaconda3/envs/tf/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.



68426 sequences remaining, 196 clusters.
=================Calculating TNF TF-IDF=================
=================Reducing Dimensionality with UMAP=================


/home/gandalf/anaconda3/envs/tf/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

/home/gandalf/anaconda3/envs/tf/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.



Embedding creation time (binning data): 487.56 seconds
 ============================ Running k-medoids clustering ============================
